In [8]:
# import
import pandas as pd
import requests
from IPython.display import display
from datetime import datetime
import plotly.express as px

In [9]:
# ETL(Extract, Transform, Load) Block:
def get_flight_data():
    # API - מאגר טיסות - EXTRACT
   
    # מזהה משאב(מגיע מהאתר)
    resource_id = "e83f763b-b7d7-479e-b172-ae981ddc6de5"

    url = f"https://data.gov.il/api/3/action/datastore_search?resource_id={resource_id}"

    # send requests to website and get response(code)
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"There is a problem: {response.status_code}")
        return None
    
    # parsing  the data we got
    data = response.json()
    
    # convert to DataFrame
    records = data['result']['records']
    df_flights = pd.DataFrame(records)
    
    #TRANSFORM

    # Rename columns to meaningful names
    df_flights = df_flights.rename(columns={
        '_id': 'id',
        'CHOPER': 'airline_code',
        'CHOPERD': 'airline_name',
        'CHFLTN': 'flight_number',
        'CHAORD': 'direction',
        'CHSTOL': 'scheduled_time',
        'CHPTOL': 'actual_time',
        'CHLOC1': 'airport_code',
        'CHLOC1D': 'airport_name',
        'CHLOC1TH': 'city_hebrew',
        'CHLOC1T': 'city_english',
        'CHLOC1CH': 'country_hebrew',
        'CHLOCCT': 'country_english',
        'CHTERM': 'terminal',
        'CHCINT': 'checkin_counters',
        'CHCKZN': 'checkin_zone',
        'CHRMINE': 'flight_status',
        'CHRMINH': 'flight_status_hebrew'
    }, errors='ignore')

    df_flights['terminal'] = pd.to_numeric(df_flights['terminal'], errors='coerce').astype('Int8')
    df_flights['flight_number'] = df_flights['flight_number'].astype(str)
    df_flights['scheduled_time'] = pd.to_datetime(df_flights['scheduled_time'])
    df_flights['actual_time'] = pd.to_datetime(df_flights['actual_time'])
    df_flights['terminal'] = pd.to_numeric(df_flights['terminal'], errors='coerce').astype('Int8')

    # missing values:
    # flights that are landing they are arrivel = mean no need to checkin so fill the empty to 'N/A'
    df_flights.loc[df_flights['direction'] == 'A', ['checkin_counters', 'checkin_zone']] = 'N/A'

    # flights that are leaving they are departure = means they do need to do check in so, if there is empty for any reason fill it as 'Missing'
    mask_missing_departures = (df_flights['direction'] == 'D') & (df_flights['checkin_counters'].isna())
    df_flights.loc[mask_missing_departures, ['checkin_counters', 'checkin_zone']] = 'Missing'
    #return to us the df - LOAD
    return df_flights


In [10]:
# update data
df_flights = get_flight_data()

# quick check for missing and duplicates rows:
# print(f"For df_flights there is: {df_flights.duplicated().sum()} duplicates rows")
# print(f"For df_flights there is: {df_flights.isna().sum()} missing values")


### EDA - Exploratory Data Analysis

In [11]:
# display the table
display(df_flights.head())

# look at the data in static way
df_flights.describe()

,id,airline_code,flight_number,airline_name,scheduled_time,actual_time,direction,airport_code,airport_name,city_hebrew,city_english,country_hebrew,country_english,terminal,checkin_counters,checkin_zone,flight_status,flight_status_hebrew
0,1,BZ,706,BLUE BIRD AIRWAYS,2026-06-20 13:55:00,2026-06-20 14:35:00,D,ATH,ATHENS,אתונה,ATHENS,יוון,GREECE,3,46-55,B,DEPARTED,המריאה
1,2,IZ,815,ARKIA ISRAELI AIRLINES,2026-06-20 14:30:00,2026-06-20 14:41:00,D,ETM,RAMON,אילת - רמון,RAMON,ישראל,ISRAEL,3,G1-G4,G,DEPARTED,המריאה
2,3,IZ,093,ARKIA ISRAELI AIRLINES,2026-06-20 14:25:00,2026-06-20 14:43:00,D,HER,HERAKLION,הרקליון,HERAKLION,יוון,GREECE,3,12-23,A,DEPARTED,המריאה
3,4,CY,110,CYPRUS AIRWAYS,2026-06-20 14:55:00,2026-06-20 14:46:00,A,LCA,LARNACA,לרנקה,LARNACA,קפריסין,CYPRUS,3,N/A,N/A,LANDED,נחתה
4,5,IZ,344,ARKIA ISRAELI AIRLINES,2026-06-20 14:35:00,2026-06-20 14:47:00,A,VRN,VILLAFRANCA-VERONA,ורונה,VERONA,איטליה,ITALY,3,N/A,N/A,LANDED,נחתה


,id,scheduled_time,actual_time,terminal
count,2434.0000,2434,2434,2434.0
mean,1217.5000,2026-06-22 16:33:19.671322880,2026-06-22 16:39:50.410846208,3.0
min,1.0000,2026-06-20 13:55:00,2026-06-20 14:35:00,3.0
25%,609.2500,2026-06-21 17:21:15,2026-06-21 17:43:45,3.0
50%,1217.5000,2026-06-22 15:50:00,2026-06-22 15:50:00,3.0
75%,1825.7500,2026-06-23 15:15:00,2026-06-23 15:15:00,3.0
max,2434.0000,2026-06-24 14:30:00,2026-06-24 14:30:00,3.0
std,702.7796,NaN,NaN,0.0


In [12]:
# create delay column:
df_flights['delay'] = (df_flights['actual_time'] - df_flights['scheduled_time']).dt.total_seconds()/60

display(df_flights.head())

,id,airline_code,flight_number,airline_name,scheduled_time,actual_time,direction,airport_code,airport_name,city_hebrew,city_english,country_hebrew,country_english,terminal,checkin_counters,checkin_zone,flight_status,flight_status_hebrew,delay
0,1,BZ,706,BLUE BIRD AIRWAYS,2026-06-20 13:55:00,2026-06-20 14:35:00,D,ATH,ATHENS,אתונה,ATHENS,יוון,GREECE,3,46-55,B,DEPARTED,המריאה,40.0
1,2,IZ,815,ARKIA ISRAELI AIRLINES,2026-06-20 14:30:00,2026-06-20 14:41:00,D,ETM,RAMON,אילת - רמון,RAMON,ישראל,ISRAEL,3,G1-G4,G,DEPARTED,המריאה,11.0
2,3,IZ,093,ARKIA ISRAELI AIRLINES,2026-06-20 14:25:00,2026-06-20 14:43:00,D,HER,HERAKLION,הרקליון,HERAKLION,יוון,GREECE,3,12-23,A,DEPARTED,המריאה,18.0
3,4,CY,110,CYPRUS AIRWAYS,2026-06-20 14:55:00,2026-06-20 14:46:00,A,LCA,LARNACA,לרנקה,LARNACA,קפריסין,CYPRUS,3,N/A,N/A,LANDED,נחתה,-9.0
4,5,IZ,344,ARKIA ISRAELI AIRLINES,2026-06-20 14:35:00,2026-06-20 14:47:00,A,VRN,VILLAFRANCA-VERONA,ורונה,VERONA,איטליה,ITALY,3,N/A,N/A,LANDED,נחתה,12.0


### The Goal of Project:
#### 1.How many flight from start of the day till now?
#### 2. 

In [13]:
# 1.How many flight from start of the day till now?
# to get the date of today
now = datetime.now()
today = now.date()

today_flight = df_flights[
    (df_flights['actual_time'].dt.date == today) &
    (df_flights['actual_time'] <= now)
]
print(f"Today is {today} was {len(today_flight)} flights")
display(today_flight.head())

Today is 2026-06-21 was 359 flights


,id,airline_code,flight_number,airline_name,scheduled_time,actual_time,direction,airport_code,airport_name,city_hebrew,city_english,country_hebrew,country_english,terminal,checkin_counters,checkin_zone,flight_status,flight_status_hebrew,delay
112,113,6H,586,ISRAIR AIRLINES,2026-06-21 00:10:00,2026-06-21 00:01:00,A,LCA,LARNACA,לרנקה,LARNACA,קפריסין,CYPRUS,3,N/A,N/A,LANDED,נחתה,-9.0
113,114,6H,893,ISRAIR AIRLINES,2026-06-20 23:00:00,2026-06-21 00:02:00,D,TBS,TBILISI,טביליסי,TBILISI,גיאורגיה,GEORGIA,3,201-205,G,DEPARTED,המריאה,62.0
114,115,LY,091,EL AL ISRAEL AIRLINES,2026-06-20 23:30:00,2026-06-21 00:04:00,D,NRT,TOKYO INTL.-NARITA,טוקיו,TOKYO,יפן,JAPAN,3,78-99,D,DEPARTED,המריאה,34.0
115,116,QF,5014,QANTAS AIRWAYS,2026-06-20 23:30:00,2026-06-21 00:04:00,D,NRT,TOKYO INTL.-NARITA,טוקיו,TOKYO,יפן,JAPAN,3,78-99,D,DEPARTED,המריאה,34.0
116,117,BZ,611,BLUE BIRD AIRWAYS,2026-06-21 00:20:00,2026-06-21 00:14:00,A,AMS,AMSTERDAM,אמסטרדם,AMSTERDAM,הולנד,NETHERLANDS,3,N/A,N/A,LANDED,נחתה,-6.0


In [31]:
# 2. Which flight company is popular
flight_company = today_flight['airline_name'].value_counts()
# print(flight_company.head(5))

# bar chart to display the popular flight company today:
# first convert flight_company into df for easy create plot:
plot_flight_company = flight_company.reset_index()


# top 5 airline - head(5)
top_airline = plot_flight_company.sort_values(by='count', ascending=False).head(5)

# columns names
top_airline.columns = ['airline_name', 'count']

#color palette:
my_custom_palette = ['#4E79A7', '#F28E2B', '#E15759', '#76B7B2', '#59A14F'] 

# bulid the chart:
fig = px.bar(
    top_airline, x='count', y='airline_name',orientation='h',color='airline_name',color_discrete_sequence=my_custom_palette,
    title="Number of flights by airline:", labels={'count':'Number of flights','airline_name':'Flight Company'})

fig.show()


In [ ]:
# 3. Which destinations are popular - Most Popular Destinations:

